# Bird Species Observation Analysis — Forest vs Grassland

This notebook covers the full pipeline for the project:

1. Load multi-sheet Excel data (Forest + Grassland)
2. Data cleaning & preprocessing
3. Exploratory Data Analysis (Temporal, Spatial, Species, Environmental, Distance/Behavior, Observer, Conservation)
4. Export a cleaned dataset for the Streamlit / Power BI dashboard

**How to run in VS Code:**
1. Keep `Bird_Monitoring_Data_FOREST.XLSX` and `Bird_Monitoring_Data_GRASSLAND.XLSX` in the same folder as this notebook.
2. Install requirements: `pip install pandas numpy matplotlib seaborn openpyxl`
3. Open this `.ipynb` file in VS Code (Jupyter extension), select a Python kernel, and Run All.


## 1. Imports & Settings

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

pd.set_option('display.max_columns', 40)
sns.set_style('whitegrid')
plt.rcParams['figure.dpi'] = 100

FOREST_FILE = 'Bird_Monitoring_Data_FOREST.XLSX'
GRASSLAND_FILE = 'Bird_Monitoring_Data_GRASSLAND.XLSX'


## 2. Load All Sheets

Each Excel file has one sheet per administrative unit (ANTI, CATO, CHOH, GWMP, HAFE, MANA, MONO, NACE, PRWI, ROCR, WOTR). We read every sheet and stack them into a single DataFrame per habitat, exactly the way `Read_multiple_excel_sheets_in_pandas.ipynb` demonstrates, then tag each row with its source sheet and habitat.

In [ ]:
def load_all_sheets(path, habitat_label):
    """Read every sheet in an Excel workbook and concatenate into one DataFrame."""
    excel_data = pd.ExcelFile(path)
    sheets_dict = {sheet: excel_data.parse(sheet) for sheet in excel_data.sheet_names}

    frames = []
    for sheet_name, df in sheets_dict.items():
        if df.empty:
            continue  # some admin-unit sheets have no records for this habitat
        df = df.copy()
        df['Admin_Unit_Code_Sheet'] = sheet_name
        frames.append(df)

    combined = pd.concat(frames, ignore_index=True)
    combined['Habitat'] = habitat_label
    return combined

forest_df = load_all_sheets(FOREST_FILE, 'Forest')
grassland_df = load_all_sheets(GRASSLAND_FILE, 'Grassland')

print('Forest sheets combined  :', forest_df.shape)
print('Grassland sheets combined:', grassland_df.shape)


## 3. Consolidate Forest + Grassland into One Dataset

The Forest file has `NPSTaxonCode` and `Site_Name`; the Grassland file has `TaxonCode` and no `Site_Name`. We align the taxon-code column name so both datasets stack cleanly into one master DataFrame.

In [ ]:
if 'NPSTaxonCode' in forest_df.columns:
    forest_df = forest_df.rename(columns={'NPSTaxonCode': 'TaxonCode'})
if 'NPSTaxonCode' in grassland_df.columns:
    grassland_df = grassland_df.rename(columns={'NPSTaxonCode': 'TaxonCode'})

df = pd.concat([forest_df, grassland_df], ignore_index=True, sort=False)
print('Combined raw shape:', df.shape)
df.head()


## 4. Data Cleaning & Preprocessing

Steps performed:
- Remove exact duplicate rows
- Strip whitespace / normalise blank strings to NaN across text columns
- Drop columns that are almost entirely empty (e.g. `Sub_Unit_Code`)
- Fill categorical missing values with an explicit label instead of leaving NaN (`Sex`, `Distance`, `ID_Method`)
- Parse `Date` into a real datetime and derive `Month`, `Month_Name`, `Season`
- Convert numeric columns (`Temperature`, `Humidity`, `AcceptedTSN`, `Visit`, `Year`) to proper numeric dtypes
- Convert TRUE/FALSE text/flags into real booleans
- Drop rows with no identified species (no `Scientific_Name`) — these are unusable for species analysis


In [ ]:
before = len(df)
df = df.drop_duplicates()
print(f'Removed {before - len(df)} exact duplicate rows')


In [ ]:
# Normalise text columns: strip whitespace, turn empty strings into real NaN
text_cols = df.select_dtypes(include='object').columns.tolist()
for c in text_cols:
    df[c] = df[c].astype(str).str.strip()
    df[c] = df[c].replace({'nan': np.nan, 'None': np.nan, '': np.nan})


In [ ]:
# Drop columns that are >90% empty (not useful for analysis)
drop_cols = [c for c in df.columns if df[c].isna().mean() > 0.9]
df = df.drop(columns=drop_cols)
print('Dropped near-empty columns:', drop_cols)


In [ ]:
# Fill meaningful defaults instead of leaving NaN
if 'Sex' in df.columns:
    df['Sex'] = df['Sex'].fillna('Undetermined')
if 'Distance' in df.columns:
    df['Distance'] = df['Distance'].fillna('Unknown')
if 'ID_Method' in df.columns:
    df['ID_Method'] = df['ID_Method'].fillna('Unknown')


In [ ]:
# Dates & derived temporal columns
df['Date'] = pd.to_datetime(df['Date'], errors='coerce')
df['Year'] = pd.to_numeric(df['Year'], errors='coerce').astype('Int64')
df['Month'] = df['Date'].dt.month
df['Month_Name'] = df['Date'].dt.month_name()

def month_to_season(m):
    if pd.isna(m):
        return np.nan
    m = int(m)
    if m in (12, 1, 2):
        return 'Winter'
    if m in (3, 4, 5):
        return 'Spring'
    if m in (6, 7, 8):
        return 'Summer'
    return 'Fall'

df['Season'] = df['Month'].apply(month_to_season)


In [ ]:
# Numeric conversions
for c in ['Temperature', 'Humidity', 'AcceptedTSN', 'Visit']:
    if c in df.columns:
        df[c] = pd.to_numeric(df[c], errors='coerce')

# Boolean conversions
bool_cols = ['Flyover_Observed', 'PIF_Watchlist_Status', 'Regional_Stewardship_Status',
             'Initial_Three_Min_Cnt', 'Previously_Obs']
for c in bool_cols:
    if c in df.columns:
        df[c] = df[c].astype(str).str.strip().str.upper().map({'TRUE': True, 'FALSE': False, '1': True, '0': False})


In [ ]:
before = len(df)
df = df.dropna(subset=['Scientific_Name']).reset_index(drop=True)
print(f'Dropped {before - len(df)} rows with no identified species')
print('Final cleaned shape:', df.shape)
df.isnull().sum()[df.isnull().sum() > 0]


In [ ]:
# Save the cleaned, analysis-ready dataset
df.to_csv('Bird_Combined_Cleaned.csv', index=False)
print('Saved: Bird_Combined_Cleaned.csv')
df.head()


---
## 5. Exploratory Data Analysis

### 5.1 Temporal Analysis — Seasonal & Yearly Trends

In [ ]:
fig, ax = plt.subplots(figsize=(7, 4))
sns.countplot(data=df, x='Season', hue='Habitat', order=['Spring', 'Summer', 'Fall', 'Winter'], ax=ax)
ax.set_title('Bird Observations by Season and Habitat')
ax.set_ylabel('Number of Observations')
plt.tight_layout()
plt.show()


In [ ]:
fig, ax = plt.subplots(figsize=(7, 4))
sns.countplot(data=df, x='Year', hue='Habitat', ax=ax)
ax.set_title('Bird Observations by Year and Habitat')
ax.set_ylabel('Number of Observations')
plt.tight_layout()
plt.show()


### 5.2 Observation Time Windows

In [ ]:
df['Start_Hour'] = pd.to_datetime(df['Start_Time'].astype(str), errors='coerce').dt.hour

fig, ax = plt.subplots(figsize=(8, 4))
sns.countplot(data=df, x='Start_Hour', hue='Habitat', ax=ax)
ax.set_title('Observation Start Hour Distribution')
ax.set_xlabel('Hour of Day')
plt.tight_layout()
plt.show()


### 5.3 Spatial Analysis — Admin Units & Plots

In [ ]:
fig, ax = plt.subplots(figsize=(9, 5))
df.groupby(['Admin_Unit_Code', 'Habitat'])['Scientific_Name'].nunique().unstack().plot(
    kind='bar', ax=ax, color=['#2c7a4b', '#c9a227'])
ax.set_title('Unique Species Count by Administrative Unit')
ax.set_ylabel('Unique Species')
plt.tight_layout()
plt.show()


In [ ]:
top_plots = df.groupby('Plot_Name')['Scientific_Name'].nunique().sort_values(ascending=False).head(10)
fig, ax = plt.subplots(figsize=(8, 5))
top_plots.plot(kind='barh', ax=ax, color='#3b6ea5')
ax.invert_yaxis()
ax.set_title('Top 10 Plots by Unique Species Count')
ax.set_xlabel('Unique Species')
plt.tight_layout()
plt.show()


### 5.4 Species Analysis — Diversity, Activity, Sex Ratio

In [ ]:
print('Total unique species:', df['Scientific_Name'].nunique())
print(df.groupby('Habitat')['Scientific_Name'].nunique())

fig, ax = plt.subplots(figsize=(8, 5))
df['Common_Name'].value_counts().head(10).plot(kind='barh', ax=ax, color='#5b8c5a')
ax.invert_yaxis()
ax.set_title('Top 10 Most Observed Species')
ax.set_xlabel('Number of Observations')
plt.tight_layout()
plt.show()


In [ ]:
fig, ax = plt.subplots(figsize=(7, 4))
sns.countplot(data=df, x='ID_Method', hue='Habitat', ax=ax,
              order=df['ID_Method'].value_counts().index)
ax.set_title('Identification Method Used')
plt.tight_layout()
plt.show()


In [ ]:
fig, ax = plt.subplots(figsize=(6, 4))
sns.countplot(data=df, x='Sex', hue='Habitat', ax=ax)
ax.set_title('Sex Ratio of Observed Birds')
plt.tight_layout()
plt.show()


### 5.5 Environmental Conditions — Weather & Disturbance

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
sns.histplot(data=df, x='Temperature', hue='Habitat', bins=30, kde=True, ax=axes[0])
axes[0].set_title('Temperature Distribution')
sns.histplot(data=df, x='Humidity', hue='Habitat', bins=30, kde=True, ax=axes[1])
axes[1].set_title('Humidity Distribution')
plt.tight_layout()
plt.show()


In [ ]:
fig, ax = plt.subplots(figsize=(8, 4))
sns.countplot(data=df, y='Sky', hue='Habitat', ax=ax, order=df['Sky'].value_counts().index)
ax.set_title('Sky Condition During Observations')
plt.tight_layout()
plt.show()


In [ ]:
fig, ax = plt.subplots(figsize=(8, 4))
sns.countplot(data=df, y='Disturbance', hue='Habitat', ax=ax, order=df['Disturbance'].value_counts().index)
ax.set_title('Disturbance Effect on Observations')
plt.tight_layout()
plt.show()


### 5.6 Distance & Behavior

In [ ]:
fig, ax = plt.subplots(figsize=(7, 4))
sns.countplot(data=df, x='Distance', hue='Habitat', ax=ax)
ax.set_title('Observation Distance from Observer')
plt.tight_layout()
plt.show()


In [ ]:
fig, ax = plt.subplots(figsize=(5, 4))
sns.countplot(data=df, x='Flyover_Observed', hue='Habitat', ax=ax)
ax.set_title('Flyover Observed Frequency')
plt.tight_layout()
plt.show()


### 5.7 Observer Trends

In [ ]:
fig, ax = plt.subplots(figsize=(7, 4))
df['Observer'].value_counts().plot(kind='bar', ax=ax, color='#8a5db3')
ax.set_title('Observations Logged per Observer')
ax.set_ylabel('Number of Observations')
plt.tight_layout()
plt.show()


In [ ]:
fig, ax = plt.subplots(figsize=(6, 4))
df.groupby('Visit')['Scientific_Name'].nunique().plot(kind='bar', ax=ax, color='#4a7fa5')
ax.set_title('Unique Species Detected by Visit Number')
ax.set_xlabel('Visit Number')
ax.set_ylabel('Unique Species')
plt.tight_layout()
plt.show()


### 5.8 Conservation Insights

In [ ]:
watchlist = df[df['PIF_Watchlist_Status'] == True]
print('Watchlist observations:', len(watchlist))
print('Watchlist species:', watchlist['Common_Name'].nunique())

fig, ax = plt.subplots(figsize=(8, 5))
watchlist['Common_Name'].value_counts().head(10).plot(kind='barh', ax=ax, color='#c0392b')
ax.invert_yaxis()
ax.set_title('Top 10 PIF-Watchlist Species Observed')
plt.tight_layout()
plt.show()


In [ ]:
fig, ax = plt.subplots(figsize=(6, 4))
sns.countplot(data=df, x='Regional_Stewardship_Status', hue='Habitat', ax=ax)
ax.set_title('Regional Stewardship Priority Species')
plt.tight_layout()
plt.show()


---
## 6. Key Findings (fill in after reviewing the charts above)

- Species diversity: Forest and Grassland habitats show roughly {{forest_species}} vs {{grassland_species}} unique species — see section 5.4.
- Seasonal trend: Most observations are concentrated in specific seasons — see section 5.1.
- Conservation: A subset of species appear on the PIF Watchlist — see section 5.8 for the priority list.
- Environmental drivers: Temperature/Humidity/Sky patterns during high-activity observations are summarised in section 5.5.

Use these findings to write the final stakeholder-facing report / dashboard narrative.

## 7. Next Step — Interactive Dashboard

A companion file `streamlit_dashboard.py` is provided to turn `Bird_Combined_Cleaned.csv` into an interactive Streamlit + Plotly dashboard. Run it with:

```bash
pip install streamlit plotly
streamlit run streamlit_dashboard.py
```
